In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
# Requires: pip install imbalanced-learn
!pip install imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = 'final_mapped_accidents.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)

df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary'], errors='ignore').select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())
X_scaled = StandardScaler().fit_transform(X_raw)

# ----------------------------------------------------------------
# STEP 1: MANUAL GRID SEARCH WITH THRESHOLDS & STD DEV
# ----------------------------------------------------------------
param_grid = {
    "clf__estimator__hidden_layer_sizes": [(8,8), (8,4), (16,8), (8,)],
    "clf__estimator__alpha": [1e-4, 1e-3],
    "sampler__sampling_strategy": [0.7, 1.0],
    "sampler__n_neighbors": [3, 5],
    "clf__threshold": [0.40, 0.45],
}

candidates = list(ParameterGrid(param_grid))
total_candidates = len(candidates)

print(f"Total Candidates: {total_candidates}")
print(f"Executing {N_REPEATS * N_SPLITS} fits per candidate (Total: {total_candidates * (N_REPEATS * N_SPLITS)})")
print("-" * 115)
print(f"{'#':<4} | {'Layers':<12} | {'Alpha':<8} | {'Samp':<5} | {'Neigh':<5} | {'Thresh':<6} | {'Bal. Acc':<10} | {'Std Dev':<9} | {'Status'}")
print("-" * 115)

results_list = []
best_score = -1
best_params_overall = None

# Manual Loop for Live Updates
for i, params in enumerate(candidates):

    # Extract specific hyperparams for this run
    mlp_args = {
        'hidden_layer_sizes': params['clf__estimator__hidden_layer_sizes'],
        'alpha': params['clf__estimator__alpha'],
        'max_iter': 500,
        'early_stopping': True,
        'random_state': RANDOM_STATE
    }

    ada_args = {
        'sampling_strategy': params['sampler__sampling_strategy'],
        'n_neighbors': params['sampler__n_neighbors'],
        'random_state': RANDOM_STATE
    }

    current_threshold = params['clf__threshold']

    # Setup CV
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    fold_scores = []

    # Inner CV loop to handle manual thresholding
    try:
        for train_idx, val_idx in cv.split(X_scaled, y):
            X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            # Balance and Train
            ada = ADASYN(**ada_args)
            X_res, y_res = ada.fit_resample(X_tr, y_tr)

            mlp = MLPClassifier(**mlp_args)
            mlp.fit(X_res, y_res)

            # Predict Probabilities
            probs = mlp.predict_proba(X_val)[:, pos_label_idx]

            # Apply CUSTOM Threshold
            preds = np.where(probs > current_threshold, pos_label_idx, other_label_idx)

            # Score
            fold_scores.append(balanced_accuracy_score(y_val, preds))

        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores, ddof=1)

    except Exception:
        mean_score = 0.0
        std_score = 0.0

    # Track Global Best
    if mean_score > best_score:
        best_score = mean_score
        best_params_overall = {
            'mlp_args': mlp_args,
            'ada_args': ada_args,
            'threshold': current_threshold,
            'std': std_score
        }

    # LIVE PRINTING
    layers_str = str(mlp_args['hidden_layer_sizes'])
    print(f"{i+1:<4} | {layers_str:<12} | {mlp_args['alpha']:<8} | {ada_args['sampling_strategy']:<5} | "
          f"{ada_args['n_neighbors']:<5} | {current_threshold:<6} | {mean_score:.4f}     | {std_score:.4f}    | "
          f"{total_candidates - (i+1)} left")

# ----------------------------------------------------------------
# STEP 2: FINAL EVALUATION OF THE WINNER
# ----------------------------------------------------------------
print("\n" + "="*50)
print(f"WINNER FOUND")
print(f"Balanced Acc: {best_score:.4f} (+/- {best_params_overall['std']:.4f})")
print(f"Threshold: {best_params_overall['threshold']}")
print(f"Layers: {best_params_overall['mlp_args']['hidden_layer_sizes']}")
print("="*50)

# Final Detailed Run for Confusion Matrix
all_y_true, all_y_pred = [], []
aggregate_cm = np.zeros((2, 2), dtype=int)

# Use the best CV splitter again for consistent evaluation
cv_eval = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

for train_idx, test_idx in cv_eval.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Apply winning Sampler
    ada_final = ADASYN(**best_params_overall['ada_args'])
    X_res, y_res = ada_final.fit_resample(X_train, y_train)

    # Apply winning MLP
    mlp_final = MLPClassifier(**best_params_overall['mlp_args'])
    mlp_final.fit(X_res, y_res)

    # Apply winning Threshold
    probs = mlp_final.predict_proba(X_test)[:, pos_label_idx]
    preds = np.where(probs > best_params_overall['threshold'], pos_label_idx, other_label_idx)

    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# Visualization
plt.figure(figsize=(7, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Final Model Results\nMean Balanced Acc: {best_score:.4f} (Threshold: {best_params_overall['threshold']})")
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
# Requires: pip install imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = 'final_mapped_accidents.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
TOP_K_MI = 20  # <-- select top 20 MI features per training fold

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)


df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx

# 2. Features (do NOT scale/fill globally to avoid leakage)
X_all = df.drop(columns=['Severity', 'Severity_Binary'], errors='ignore').select_dtypes(include=[np.number])
# note: we do not .fillna() or .scale() here globally; per-fold handling below

# ----------------------------------------------------------------
# STEP 1: MANUAL GRID SEARCH WITH THRESHOLDS & STD DEV
# ----------------------------------------------------------------
param_grid = {
    "clf__estimator__hidden_layer_sizes": [(8,8), (8,4), (16,8), (8,)],
    "clf__estimator__alpha": [1e-4, 1e-3],
    "sampler__sampling_strategy": [0.7, 1.0],
    "sampler__n_neighbors": [3, 5],
    "clf__threshold": [0.40, 0.45],
}

candidates = list(ParameterGrid(param_grid))
total_candidates = len(candidates)

print(f"Total Candidates: {total_candidates}")
print(f"Executing {N_REPEATS * N_SPLITS} fits per candidate (Total: {total_candidates * (N_REPEATS * N_SPLITS)})")
print("-" * 115)
print(f"{'#':<4} | {'Layers':<12} | {'Alpha':<8} | {'Samp':<5} | {'Neigh':<5} | {'Thresh':<6} | {'Bal. Acc':<10} | {'Std Dev':<9} | {'Status'}")
print("-" * 115)

results_list = []
best_score = -1
best_params_overall = None

# Manual Loop for Live Updates
for i, params in enumerate(candidates):

    # Extract specific hyperparams for this run
    mlp_args = {
        'hidden_layer_sizes': params['clf__estimator__hidden_layer_sizes'],
        'alpha': params['clf__estimator__alpha'],
        'max_iter': 500,
        'early_stopping': True,
        'random_state': RANDOM_STATE
    }

    ada_args = {
        'sampling_strategy': params['sampler__sampling_strategy'],
        'n_neighbors': params['sampler__n_neighbors'],
        'random_state': RANDOM_STATE
    }

    current_threshold = params['clf__threshold']

    # Setup CV
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    fold_scores = []

    # Inner CV loop to handle manual thresholding and per-fold MI selection
    try:
        for train_idx, val_idx in cv.split(X_all, y):
            # Get raw train/val feature tables (no global fills)
            X_tr_raw = X_all.iloc[train_idx].copy()
            X_val_raw = X_all.iloc[val_idx].copy()
            y_tr, y_val = y[train_idx], y[val_idx]

            # -------------------------
            # 1) Impute using training medians (train-only)
            # -------------------------
            medians = X_tr_raw.median()
            X_tr = X_tr_raw.fillna(medians).fillna(0)
            X_val = X_val_raw.fillna(medians).fillna(0)

            # -------------------------
            # 2) (Optional) Density filter - keep columns with >10% non-zero
            # -------------------------
            dense_cols = X_tr.columns[(X_tr > 0).mean() > 0.10]
            if len(dense_cols) == 0:
                dense_cols = X_tr.columns  # fallback
            X_tr_dense = X_tr[dense_cols]
            X_val_dense = X_val[dense_cols]

            # -------------------------
            # 3) Mutual Information Selection (TRAIN ONLY) - select top 20
            # -------------------------
            if X_tr_dense.shape[1] == 0:
                # nothing to train on: give neutral score
                fold_scores.append(0.5)
                continue

            mi_scores = mutual_info_classif(X_tr_dense, y_tr, random_state=RANDOM_STATE)
            mi_series = pd.Series(mi_scores, index=X_tr_dense.columns).sort_values(ascending=False)
            top_k = min(TOP_K_MI, len(mi_series))
            selected_features = mi_series.head(top_k).index.tolist()
            print(selected_features)

            X_tr_sel = X_tr_dense[selected_features]
            X_val_sel = X_val_dense[selected_features]

            # -------------------------
            # 4) Scaling (TRAIN ONLY)
            # -------------------------
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr_sel)
            X_val_scaled = scaler.transform(X_val_sel)

            # -------------------------
            # 5) ADASYN + MLP (train on resampled training set)
            # -------------------------
            ada = ADASYN(**ada_args)
            X_res, y_res = ada.fit_resample(X_tr_scaled, y_tr)

            mlp = MLPClassifier(**mlp_args)
            mlp.fit(X_res, y_res)

            # Predict Probabilities
            probs = mlp.predict_proba(X_val_scaled)[:, pos_label_idx]

            # Apply CUSTOM Threshold
            preds = np.where(probs > current_threshold, pos_label_idx, other_label_idx)

            # Score
            fold_scores.append(balanced_accuracy_score(y_val, preds))

        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores, ddof=1) if len(fold_scores) > 1 else 0.0

    except Exception:
        mean_score = 0.0
        std_score = 0.0

    # Track Global Best
    if mean_score > best_score:
        best_score = mean_score
        best_params_overall = {
            'mlp_args': mlp_args,
            'ada_args': ada_args,
            'threshold': current_threshold,
            'std': std_score
        }

    # LIVE PRINTING
    layers_str = str(mlp_args['hidden_layer_sizes'])
    print(f"{i+1:<4} | {layers_str:<12} | {mlp_args['alpha']:<8} | {ada_args['sampling_strategy']:<5} | "
          f"{ada_args['n_neighbors']:<5} | {current_threshold:<6} | {mean_score:.4f}     | {std_score:.4f}    | "
          f"{total_candidates - (i+1)} left")

# ----------------------------------------------------------------
# STEP 2: FINAL EVALUATION OF THE WINNER (with per-fold top-20 MI selection)
# ----------------------------------------------------------------
print("\n" + "="*50)
print(f"WINNER FOUND")
print(f"Balanced Acc: {best_score:.4f} (+/- {best_params_overall['std']:.4f})")
print(f"Threshold: {best_params_overall['threshold']}")
print(f"Layers: {best_params_overall['mlp_args']['hidden_layer_sizes']}")
print("="*50)

# Final Detailed Run for Confusion Matrix
all_y_true, all_y_pred = [], []
aggregate_cm = np.zeros((2, 2), dtype=int)

# Use the same CV splitter again for consistent evaluation
cv_eval = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

for train_idx, test_idx in cv_eval.split(X_all, y):
    X_tr_raw = X_all.iloc[train_idx].copy()
    X_te_raw = X_all.iloc[test_idx].copy()
    y_tr, y_te = y[train_idx], y[test_idx]

    # Impute
    medians = X_tr_raw.median()
    X_tr = X_tr_raw.fillna(medians).fillna(0)
    X_te = X_te_raw.fillna(medians).fillna(0)

    # Density filter
    dense_cols = X_tr.columns[(X_tr > 0).mean() > 0.10]
    if len(dense_cols) == 0:
        dense_cols = X_tr.columns
    X_tr_dense = X_tr[dense_cols]
    X_te_dense = X_te[dense_cols]

    # MI selection on training fold
    mi_scores = mutual_info_classif(X_tr_dense, y_tr, random_state=RANDOM_STATE)
    mi_series = pd.Series(mi_scores, index=X_tr_dense.columns).sort_values(ascending=False)
    top_k = min(TOP_K_MI, len(mi_series))
    selected_features = mi_series.head(top_k).index.tolist()

    X_tr_sel = X_tr_dense[selected_features]
    X_te_sel = X_te_dense[selected_features]

    # Scale
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_sel)
    X_te_scaled = scaler.transform(X_te_sel)

    # ADASYN + MLP using best params
    ada_final = ADASYN(**best_params_overall['ada_args'])
    X_res, y_res = ada_final.fit_resample(X_tr_scaled, y_tr)

    mlp_final = MLPClassifier(**best_params_overall['mlp_args'])
    mlp_final.fit(X_res, y_res)

    probs = mlp_final.predict_proba(X_te_scaled)[:, pos_label_idx]
    preds = np.where(probs > best_params_overall['threshold'], pos_label_idx, other_label_idx)

    all_y_true.extend(y_te)
    all_y_pred.extend(preds)
    aggregate_cm += confusion_matrix(y_te, preds, labels=[0, 1])

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# Visualization
plt.figure(figsize=(7, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Final Model Results\nMean Balanced Acc: {best_score:.4f} (Threshold: {best_params_overall['threshold']})")
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
# Requires: pip install imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = 'final_merged_cleaned_data_no_duplicates.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)

# --- Parse Date + Time and Extract Features ---
# Using the provided 'date' and 'time' columns
df['dt_combined'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
df['hour'] = df['dt_combined'].dt.hour
df['month'] = df['dt_combined'].dt.month

# Fill timestamp gaps if any with the mode
for col in ['hour', 'month']:
    df[col] = df[col].fillna(df[col].mode()[0])

df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx

# 2. Features Preprocessing
# List of columns to treat as categorical (One-Hot)
CAT_FEATURES = [
    'first_road_class', 'road_type', 'junction_detail', 'second_road_class',
    'pedestrian_crossing', 'light_conditions', 'weather_conditions',
    'road_surface_conditions', 'special_conditions_at_site',
    'carriageway_hazards', 'urban_or_rural_area', 'trunk_road_flag',
    'day_of_week', 'hour', 'month'
]

# Convert categorical features to string and One-Hot encode
X_cat = df[CAT_FEATURES].astype(str)
X_cat_encoded = pd.get_dummies(X_cat, drop_first=True)

# Identify Numeric Features (Excluding target, dates, and raw categorical strings)
# Include 'longitude', 'latitude', 'speed_limit', 'collision_year'
NUM_FEATURES = ['longitude', 'latitude', 'speed_limit', 'collision_year']
X_num = df[NUM_FEATURES].fillna(df[NUM_FEATURES].median())

# Combine and Scale
X_combined = pd.concat([X_num, X_cat_encoded], axis=1)
X_scaled = StandardScaler().fit_transform(X_combined)

print(f"Feature set size: {X_combined.shape[1]} (Numeric + One-Hot Encoded Categories)")

# ----------------------------------------------------------------
# STEP 1: MANUAL GRID SEARCH WITH THRESHOLDS & STD DEV
# ----------------------------------------------------------------
param_grid = {
    "clf__estimator__hidden_layer_sizes": [(8,8), (8,4), (16,8), (8,)],
    "clf__estimator__alpha": [1e-4, 1e-3],
    "sampler__sampling_strategy": [0.7, 1.0],
    "sampler__n_neighbors": [3, 5],
    "clf__threshold": [0.40, 0.45],
}

candidates = list(ParameterGrid(param_grid))
total_candidates = len(candidates)

print(f"Total Candidates: {total_candidates}")
print(f"Executing {N_REPEATS * N_SPLITS} fits per candidate (Total: {total_candidates * (N_REPEATS * N_SPLITS)})")
print("-" * 115)
print(f"{'#':<4} | {'Layers':<12} | {'Alpha':<8} | {'Samp':<5} | {'Neigh':<5} | {'Thresh':<6} | {'Bal. Acc':<10} | {'Std Dev':<9} | {'Status'}")
print("-" * 115)

results_list = []
best_score = -1
best_params_overall = None

# Manual Loop for Live Updates
for i, params in enumerate(candidates):

    # Extract specific hyperparams for this run
    mlp_args = {
        'hidden_layer_sizes': params['clf__estimator__hidden_layer_sizes'],
        'alpha': params['clf__estimator__alpha'],
        'max_iter': 500,
        'early_stopping': True,
        'random_state': RANDOM_STATE
    }

    ada_args = {
        'sampling_strategy': params['sampler__sampling_strategy'],
        'n_neighbors': params['sampler__n_neighbors'],
        'random_state': RANDOM_STATE
    }

    current_threshold = params['clf__threshold']

    # Setup CV
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    fold_scores = []

    # Inner CV loop to handle manual thresholding
    try:
        for train_idx, val_idx in cv.split(X_scaled, y):
            X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            # Balance and Train
            ada = ADASYN(**ada_args)
            X_res, y_res = ada.fit_resample(X_tr, y_tr)

            mlp = MLPClassifier(**mlp_args)
            mlp.fit(X_res, y_res)

            # Predict Probabilities
            probs = mlp.predict_proba(X_val)[:, pos_label_idx]

            # Apply CUSTOM Threshold
            preds = np.where(probs > current_threshold, pos_label_idx, other_label_idx)

            # Score
            fold_scores.append(balanced_accuracy_score(y_val, preds))

        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores, ddof=1)

    except Exception:
        mean_score = 0.0
        std_score = 0.0

    # Track Global Best
    if mean_score > best_score:
        best_score = mean_score
        best_params_overall = {
            'mlp_args': mlp_args,
            'ada_args': ada_args,
            'threshold': current_threshold,
            'std': std_score
        }

    # LIVE PRINTING
    layers_str = str(mlp_args['hidden_layer_sizes'])
    print(f"{i+1:<4} | {layers_str:<12} | {mlp_args['alpha']:<8} | {ada_args['sampling_strategy']:<5} | "
          f"{ada_args['n_neighbors']:<5} | {current_threshold:<6} | {mean_score:.4f}     | {std_score:.4f}    | "
          f"{total_candidates - (i+1)} left")

# ----------------------------------------------------------------
# STEP 2: FINAL EVALUATION OF THE WINNER
# ----------------------------------------------------------------
print("\n" + "="*50)
print(f"WINNER FOUND")
print(f"Balanced Acc: {best_score:.4f} (+/- {best_params_overall['std']:.4f})")
print(f"Threshold: {best_params_overall['threshold']}")
print(f"Layers: {best_params_overall['mlp_args']['hidden_layer_sizes']}")
print("="*50)

# Final Detailed Run for Confusion Matrix
all_y_true, all_y_pred = [], []
aggregate_cm = np.zeros((2, 2), dtype=int)

cv_eval = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

for train_idx, test_idx in cv_eval.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Apply winning Sampler
    ada_final = ADASYN(**best_params_overall['ada_args'])
    X_res, y_res = ada_final.fit_resample(X_train, y_train)

    # Apply winning MLP
    mlp_final = MLPClassifier(**best_params_overall['mlp_args'])
    mlp_final.fit(X_res, y_res)

    # Apply winning Threshold
    probs = mlp_final.predict_proba(X_test)[:, pos_label_idx]
    preds = np.where(probs > best_params_overall['threshold'], pos_label_idx, other_label_idx)

    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# Visualization
plt.figure(figsize=(7, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Final Model Results\nMean Balanced Acc: {best_score:.4f} (Threshold: {best_params_overall['threshold']})")
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, balanced_accuracy_score
# Requires: pip install imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
CSV_PATH = 'final_mapped_accidents.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
TOP_K_MI = 20

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)

df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx

# ----------------------------------------------------------------
# NEW: GLOBAL FEATURE SELECTION (EXTRACT ONCE)
# ----------------------------------------------------------------
X_init = df.drop(columns=['Severity', 'Severity_Binary'], errors='ignore').select_dtypes(include=[np.number])

# Temporary preparation for MI calculation
X_mi_temp = X_init.fillna(X_init.median()).fillna(0)
dense_cols = X_mi_temp.columns[(X_mi_temp > 0).mean() > 0.10]
if len(dense_cols) == 0:
    dense_cols = X_mi_temp.columns
X_mi_temp = X_mi_temp[dense_cols]

print(f"Calculating Mutual Information once for all features...")
mi_scores = mutual_info_classif(X_mi_temp, y, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=X_mi_temp.columns).sort_values(ascending=False)
top_k = min(TOP_K_MI, len(mi_series))
SELECTED_FEATURES = mi_series.head(top_k).index.tolist()

print(f"Features selected: {SELECTED_FEATURES}")

# Filter X_all to only use these features globally
X_all = X_init[SELECTED_FEATURES]

# ----------------------------------------------------------------
# STEP 1: MANUAL GRID SEARCH WITH THRESHOLDS & STD DEV
# ----------------------------------------------------------------
param_grid = {
    "clf__estimator__hidden_layer_sizes": [(8,8), (8,4), (16,8), (8,)],
    "clf__estimator__alpha": [1e-4, 1e-3],
    "sampler__sampling_strategy": [0.7, 1.0],
    "sampler__n_neighbors": [3, 5],
    "clf__threshold": [0.40, 0.45],
}

candidates = list(ParameterGrid(param_grid))
total_candidates = len(candidates)

print(f"Total Candidates: {total_candidates}")
print(f"Executing {N_REPEATS * N_SPLITS} fits per candidate (Total: {total_candidates * (N_REPEATS * N_SPLITS)})")
print("-" * 115)
print(f"{'#':<4} | {'Layers':<12} | {'Alpha':<8} | {'Samp':<5} | {'Neigh':<5} | {'Thresh':<6} | {'Bal. Acc':<10} | {'Std Dev':<9} | {'Status'}")
print("-" * 115)

results_list = []
best_score = -1
best_params_overall = None

# Manual Loop for Live Updates
for i, params in enumerate(candidates):

    mlp_args = {
        'hidden_layer_sizes': params['clf__estimator__hidden_layer_sizes'],
        'alpha': params['clf__estimator__alpha'],
        'max_iter': 500,
        'early_stopping': True,
        'random_state': RANDOM_STATE
    }

    ada_args = {
        'sampling_strategy': params['sampler__sampling_strategy'],
        'n_neighbors': params['sampler__n_neighbors'],
        'random_state': RANDOM_STATE
    }

    current_threshold = params['clf__threshold']

    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    fold_scores = []

    try:
        for train_idx, val_idx in cv.split(X_all, y):
            X_tr_raw = X_all.iloc[train_idx].copy()
            X_val_raw = X_all.iloc[val_idx].copy()
            y_tr, y_val = y[train_idx], y[val_idx]

            # Impute using training medians
            medians = X_tr_raw.median()
            X_tr = X_tr_raw.fillna(medians).fillna(0)
            X_val = X_val_raw.fillna(medians).fillna(0)

            # Scaling
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)

            # ADASYN + MLP
            ada = ADASYN(**ada_args)
            X_res, y_res = ada.fit_resample(X_tr_scaled, y_tr)

            mlp = MLPClassifier(**mlp_args)
            mlp.fit(X_res, y_res)

            probs = mlp.predict_proba(X_val_scaled)[:, pos_label_idx]
            preds = np.where(probs > current_threshold, pos_label_idx, other_label_idx)

            fold_scores.append(balanced_accuracy_score(y_val, preds))

        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores, ddof=1) if len(fold_scores) > 1 else 0.0

    except Exception:
        mean_score = 0.0
        std_score = 0.0

    if mean_score > best_score:
        best_score = mean_score
        best_params_overall = {
            'mlp_args': mlp_args,
            'ada_args': ada_args,
            'threshold': current_threshold,
            'std': std_score
        }

    # LIVE PRINTING (Restored)
    layers_str = str(mlp_args['hidden_layer_sizes'])
    print(f"{i+1:<4} | {layers_str:<12} | {mlp_args['alpha']:<8} | {ada_args['sampling_strategy']:<5} | "
          f"{ada_args['n_neighbors']:<5} | {current_threshold:<6} | {mean_score:.4f}     | {std_score:.4f}    | "
          f"{total_candidates - (i+1)} left")

# ----------------------------------------------------------------
# STEP 2: FINAL EVALUATION (Using global features)
# ----------------------------------------------------------------
print("\n" + "="*50)
print(f"WINNER FOUND")
print(f"Balanced Acc: {best_score:.4f} (+/- {best_params_overall['std']:.4f})")
print(f"Threshold: {best_params_overall['threshold']}")
print(f"Layers: {best_params_overall['mlp_args']['hidden_layer_sizes']}")
print("="*50)

all_y_true, all_y_pred = [], []
aggregate_cm = np.zeros((2, 2), dtype=int)

cv_eval = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

for train_idx, test_idx in cv_eval.split(X_all, y):
    X_tr_raw = X_all.iloc[train_idx].copy()
    X_te_raw = X_all.iloc[test_idx].copy()
    y_tr, y_te = y[train_idx], y[test_idx]

    medians = X_tr_raw.median()
    X_tr = X_tr_raw.fillna(medians).fillna(0)
    X_te = X_te_raw.fillna(medians).fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    ada_final = ADASYN(**best_params_overall['ada_args'])
    X_res, y_res = ada_final.fit_resample(X_tr_scaled, y_tr)

    mlp_final = MLPClassifier(**best_params_overall['mlp_args'])
    mlp_final.fit(X_res, y_res)

    probs = mlp_final.predict_proba(X_te_scaled)[:, pos_label_idx]
    preds = np.where(probs > best_params_overall['threshold'], pos_label_idx, other_label_idx)

    all_y_true.extend(y_te)
    all_y_pred.extend(preds)
    aggregate_cm += confusion_matrix(y_te, preds, labels=[0, 1])

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# Visualization
plt.figure(figsize=(7, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Final Model Results\nMean Balanced Acc: {best_score:.4f} (Threshold: {best_params_overall['threshold']})")
plt.ylabel('True Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()